
---

## 📚 Sobre este Material

Este material ha sido diseñado con el propósito de **capacitar, actualizar y practicar** conceptos fundamentales de Markdown en Jupyter Notebook. Es una herramienta pensada para facilitar el aprendizaje y la documentación efectiva de proyectos de análisis de datos y ciencia de datos.

### 🤝 Compartir y Colaborar

Este contenido es **libre para compartir, revisar, divulgar y mejorar**. Se promueve activamente su distribución en la comunidad para que más personas puedan beneficiarse y contribuir a su mejora continua. Tu feedback y sugerencias son siempre bienvenidos.

### 👨‍💻 Autor

**Andrés Muñoz**  
*AI & Data Strategy Leader passionate about NLP, LLMs, and MLOps. Driving innovation with data*

- 💼 LinkedIn: [in/amms1989](https://linkedin.com/in/amms1989)
- 🐙 GitHub: [https://github.com/anguihero](https://github.com/anguihero)

---

# Sesión 07: Estadística Aplicada y Preprocesamiento de Datos

**Autor:** anmmunozsa@outlook.es · Material de código abierto para compartir y aprender colectivamente.

## 🎯 Objetivo de la sesión
Interpretar estadísticamente un dataset (tendencia central, dispersión, distribución, correlación) y dejarlo listo para modelar.

## 🗺️ Tabla de Contenido
1. [Introducción](#intro)
2. [Carga del dataset: Credit EDA Case Study](#carga)
3. [Estadística descriptiva](#descriptiva)
4. [Distribuciones de probabilidad](#distribuciones)
5. [Correlación: Pearson vs. Spearman](#correlacion)
6. [Detección de outliers con IQR](#outliers)
7. [Valores faltantes: estrategias de imputación](#faltantes)
8. [Codificación de variables categóricas](#encoding)
9. [Escalado de variables numéricas](#escalado)
10. [Ejemplos de aplicación real](#aplicaciones)
11. [Retos de práctica](#retos)


<a id="intro"></a>
## 1. Introducción (para dummies)

Antes de entrenar cualquier modelo, hay que **entender** y **preparar** los datos. Esta sesión responde dos preguntas: "¿qué me dicen los números?" (estadística) y "¿cómo dejo los datos listos para un algoritmo?" (preprocesamiento). Se estima que esto ocupa **60-80% del tiempo real** de un proyecto de ciencia de datos.

<a id="carga"></a>
## 2. Carga del Dataset: Credit EDA Case Study

Usaremos `application_data.csv` del dataset [Credit EDA Case Study](https://www.kaggle.com/datasets/venkatasubramanian/credit-eda-case-study) de Kaggle, sobre solicitudes de crédito. Es un dataset grande (~300k filas, 122 columnas), así que trabajaremos con un **subconjunto de columnas relevantes** y una **muestra** para que todo corra ágil en clase.

In [ ]:
import kagglehub
import os
import pandas as pd
import numpy as np

path = kagglehub.dataset_download("venkatasubramanian/credit-eda-case-study")
print(os.listdir(path))

columnas_utiles = [
    "TARGET", "NAME_CONTRACT_TYPE", "CODE_GENDER", "AMT_INCOME_TOTAL",
    "AMT_CREDIT", "AMT_ANNUITY", "NAME_EDUCATION_TYPE", "NAME_FAMILY_STATUS",
    "DAYS_BIRTH", "DAYS_EMPLOYED", "OCCUPATION_TYPE",
]


In [ ]:
df = pd.read_csv(
    os.path.join(path, "application_data.csv"),
    usecols=columnas_utiles,
).sample(5000, random_state=42).reset_index(drop=True)

# DAYS_BIRTH y DAYS_EMPLOYED vienen negativos (días hacia atrás desde hoy) -> convertimos a años positivos
df["edad_anios"] = (-df["DAYS_BIRTH"] / 365).round(1)
df["antiguedad_laboral_anios"] = (-df["DAYS_EMPLOYED"] / 365).round(1)

df.head()


> 💡 **Alternativa más liviana:** si el dataset de crédito tarda demasiado en tu conexión, puedes practicar la misma teoría con `from sklearn.datasets import load_iris` (pequeño, sin faltantes) y luego volver a aplicar las técnicas sobre `df` cuando tengas tiempo.

<a id="descriptiva"></a>
## 3. Estadística Descriptiva

### 🔬 Teoría técnica
- **Media:** sensible a outliers.
- **Mediana:** robusta ante outliers, mejor para variables como ingresos.
- **Moda:** el valor más frecuente, útil en categóricas.
- **Varianza / Desviación estándar:** qué tan dispersos están los datos.
- **IQR (Q3 - Q1):** dispersión del 50% central, base para detectar outliers.

In [ ]:
df[["AMT_INCOME_TOTAL", "AMT_CREDIT", "edad_anios"]].describe()

In [ ]:
ingreso = df["AMT_INCOME_TOTAL"]

print("Media:", ingreso.mean())
print("Mediana:", ingreso.median())
print("Moda:", ingreso.mode()[0])
print("Desviación estándar:", ingreso.std())

q1, q3 = ingreso.quantile([0.25, 0.75])
iqr = q3 - q1
print(f"Q1={q1}, Q3={q3}, IQR={iqr}")

### 🧠 Resumen para dummies
Si la media y la mediana son muy distintas, hay outliers o la distribución está sesgada — mira siempre ambas, nunca confíes en una sola.

<a id="distribuciones"></a>
## 4. Distribuciones de Probabilidad

### 🔬 Teoría técnica
La distribución **Normal** (campana de Gauss) es la más común en estadística. La **regla empírica 68-95-99.7** dice que, en una distribución normal, el 68% de los datos cae dentro de ±1 desviación estándar de la media, el 95% dentro de ±2, y el 99.7% dentro de ±3.

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(6, 4))
ax.hist(df["edad_anios"], bins=25, color="teal", edgecolor="white")
ax.axvline(df["edad_anios"].mean(), color="red", linestyle="--", label="Media")
ax.set_title("Distribución de la Edad de los Solicitantes")
ax.legend()
plt.show()

### 🧠 Resumen para dummies
No todas las variables reales son perfectamente normales (ingresos casi nunca lo son: tienen cola larga a la derecha), pero la regla empírica es un buen punto de referencia rápido cuando sí se aproximan.

<a id="correlacion"></a>
## 5. Correlación: Pearson vs. Spearman

### 🔬 Teoría técnica
- **Pearson:** mide relación **lineal** entre dos variables numéricas.
- **Spearman:** mide relación **monótona** (basada en rangos), más robusta cuando la relación no es estrictamente lineal o hay outliers.

⚠️ Correlación **no implica causalidad**.

In [ ]:
import seaborn as sns

numericas = df[["AMT_INCOME_TOTAL", "AMT_CREDIT", "AMT_ANNUITY", "edad_anios", "antiguedad_laboral_anios"]]

corr_pearson = numericas.corr(method="pearson")
corr_spearman = numericas.corr(method="spearman")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.heatmap(corr_pearson, annot=True, cmap="coolwarm", vmin=-1, vmax=1, ax=axes[0])
axes[0].set_title("Correlación de Pearson")
sns.heatmap(corr_spearman, annot=True, cmap="coolwarm", vmin=-1, vmax=1, ax=axes[1])
axes[1].set_title("Correlación de Spearman")
plt.tight_layout()
plt.show()

### 🧠 Resumen para dummies
Si Pearson y Spearman dan resultados muy distintos para el mismo par de variables, es señal de que la relación no es lineal — no descartes la variable, solo no esperes que un modelo puramente lineal la capture bien.

<a id="outliers"></a>
## 6. Detección de Outliers con IQR

### 🔬 Teoría técnica
Un valor se considera outlier si cae por debajo de `Q1 - 1.5*IQR` o por encima de `Q3 + 1.5*IQR`. Este es el criterio detrás del boxplot.

In [ ]:
def detectar_outliers_iqr(serie):
    q1, q3 = serie.quantile([0.25, 0.75])
    iqr = q3 - q1
    limite_inferior = q1 - 1.5 * iqr
    limite_superior = q3 + 1.5 * iqr
    return serie[(serie < limite_inferior) | (serie > limite_superior)]


outliers_ingreso = detectar_outliers_iqr(df["AMT_INCOME_TOTAL"])
print(f"Se encontraron {len(outliers_ingreso)} outliers de {len(df)} registros ({len(outliers_ingreso)/len(df):.1%})")

sns.boxplot(x=df["AMT_INCOME_TOTAL"])
plt.title("Boxplot de Ingresos (los puntos son outliers)")
plt.show()

### 🧠 Resumen para dummies
Un outlier no siempre es un error — puede ser un caso real e importante (ej. un solicitante con ingresos muy altos). Antes de eliminarlo, investiga si es un dato inválido o un caso legítimo.

<a id="faltantes"></a>
## 7. Valores Faltantes: Estrategias de Imputación

### 🔬 Teoría técnica

| Estrategia | Cuándo usarla |
|---|---|
| Eliminar filas | Pocos faltantes (<5%) |
| Eliminar columnas | Columna con >50% de faltantes |
| Imputar con media | Numérica, distribución simétrica |
| Imputar con mediana | Numérica, distribución sesgada u outliers |
| Imputar con moda | Categórica |
| Imputar con constante | Cuando "faltante" tiene significado propio |
| Imputación predictiva (KNN) | Cuando hay patrones y suficientes datos |

⚠️ En un flujo real, la imputación se calcula **solo con datos de entrenamiento** (se profundiza en la Sesión 13 con `Pipeline`).

In [ ]:
print(df.isna().mean().sort_values(ascending=False))

In [ ]:
from sklearn.impute import SimpleImputer

df_imputado = df.copy()

# Numérica: imputar con la mediana (AMT_ANNUITY puede tener outliers)
imputador_num = SimpleImputer(strategy="median")
df_imputado[["AMT_ANNUITY"]] = imputador_num.fit_transform(df_imputado[["AMT_ANNUITY"]])

# Categórica: imputar con la moda (valor más frecuente)
imputador_cat = SimpleImputer(strategy="most_frequent")
df_imputado[["OCCUPATION_TYPE"]] = imputador_cat.fit_transform(df_imputado[["OCCUPATION_TYPE"]])

print(df_imputado.isna().sum())

<a id="encoding"></a>
## 8. Codificación de Variables Categóricas

### 🔬 Teoría técnica
- **One-Hot Encoding:** para variables **nominales** (sin orden), crea una columna binaria por categoría.
- **Label/Ordinal Encoding:** para variables **ordinales** (con orden natural), asigna números respetando el orden.

In [ ]:
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder

# One-Hot para NAME_CONTRACT_TYPE (nominal: no hay orden entre "Cash loans" y "Revolving loans")
ohe = OneHotEncoder(sparse_output=False, drop="first")
contrato_ohe = ohe.fit_transform(df_imputado[["NAME_CONTRACT_TYPE"]])
print("Categorías detectadas:", ohe.categories_)
print(contrato_ohe[:5])

# Ordinal para NAME_EDUCATION_TYPE (sí hay un orden natural de nivel educativo)
orden_educacion = [[
    "Lower secondary", "Secondary / secondary special",
    "Incomplete higher", "Higher education", "Academic degree",
]]
oe = OrdinalEncoder(categories=orden_educacion, handle_unknown="use_encoded_value", unknown_value=-1)
df_imputado["educacion_ordinal"] = oe.fit_transform(df_imputado[["NAME_EDUCATION_TYPE"]])
df_imputado[["NAME_EDUCATION_TYPE", "educacion_ordinal"]].drop_duplicates()

### 🧠 Resumen para dummies
Pregúntate: "¿estas categorías tienen un orden lógico?". Si sí → Ordinal/Label. Si no → One-Hot.

<a id="escalado"></a>
## 9. Escalado de Variables Numéricas

### 🔬 Teoría técnica
| Método | Fórmula (idea) | Cuándo usarlo |
|---|---|---|
| **Normalización (Min-Max)** | Lleva todo a [0, 1] | Datos con límites conocidos, sin outliers extremos |
| **Estandarización (Z-score)** | Media 0, desviación estándar 1 | La opción por defecto para la mayoría de algoritmos |
| **Escalado Robusto** | Usa mediana e IQR | Cuando hay outliers importantes |

In [ ]:
from sklearn.preprocessing import MinMaxScaler, StandardScaler, RobustScaler

variables_numericas = df_imputado[["AMT_INCOME_TOTAL", "AMT_CREDIT", "AMT_ANNUITY"]]

escalado_std = StandardScaler().fit_transform(variables_numericas)
escalado_robusto = RobustScaler().fit_transform(variables_numericas)

print("Media tras estandarizar (debería ser ~0):", escalado_std.mean(axis=0).round(4))
print("Ejemplo escalado robusto (primeras 3 filas):\n", escalado_robusto[:3])

### 🧠 Resumen para dummies
Árboles de decisión, Random Forest y XGBoost **no** necesitan escalado. Regresión Logística, KNN, SVM y PCA **sí** lo necesitan — lo verás en acción en las próximas sesiones.

## 🔎 Laboratorio de profundización: estadística por tipo de variable

No todas las variables se resumen igual:

- Numérica: media, mediana, desviación, IQR y cuantiles.
- Categórica: frecuencias, proporciones, moda y cardinalidad.
- Ordinal: conserva orden, pero la distancia entre niveles no necesariamente es uniforme.
- Fecha: rango, frecuencia y componentes temporales.

La imputación, codificación y escala son **parámetros aprendidos**: deben ajustarse solo con entrenamiento para evitar fuga de información.


In [ ]:
# Paso 1: comparar medidas sensibles y robustas
serie = pd.Series([10, 11, 12, 12, 13, 14, 120], name="monto")
q1, q3 = serie.quantile([0.25, 0.75])
iqr = q3 - q1
print({
    "media": serie.mean(),
    "mediana": serie.median(),
    "desviacion": serie.std(),
    "IQR": iqr,
})


In [ ]:
# Paso 2: propiedades de una variable categórica
segmentos = pd.Series(["A", "B", "A", None, "C", "A"], dtype="category")
print("Cardinalidad:", segmentos.nunique(dropna=False))
print(segmentos.value_counts(dropna=False, normalize=True))


In [ ]:
# Paso 3: fit aprende; transform aplica lo aprendido
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

X_demo = pd.DataFrame({"edad": [22, 35, np.nan, 48], "ingreso": [1.2, 2.8, 2.1, 4.5]})
X_entrena, X_nuevo = X_demo.iloc[:3], X_demo.iloc[3:]

imputador = SimpleImputer(strategy="median")
escalador = StandardScaler()
X_entrena_imp = imputador.fit_transform(X_entrena)
X_entrena_std = escalador.fit_transform(X_entrena_imp)
X_nuevo_std = escalador.transform(imputador.transform(X_nuevo))
print("Medianas aprendidas:", imputador.statistics_)
print("Medias aprendidas:", escalador.mean_)
print("Dato nuevo transformado:", X_nuevo_std)


### Fórmulas y parámetros básicos

Estandarización:

$$z = \frac{x-\mu_{train}}{\sigma_{train}}$$

IQR:

$$IQR=Q_3-Q_1,\qquad [Q_1-1.5IQR,\ Q_3+1.5IQR]$$

Hiperparámetros de preprocesamiento incluyen `strategy` del imputador, categorías de `OneHotEncoder`, `handle_unknown` y el rango de `MinMaxScaler`. No son decisiones neutrales: deben justificarse con el significado de la variable y el modelo posterior.


<a id="aplicaciones"></a>
## 10. Ejemplos de Aplicación en el Mundo Real

- Un analista de riesgo crediticio usa IQR para detectar solicitudes con ingresos reportados anómalamente altos (posible fraude).
- Un banco decide si excluir o no una variable con 60% de datos faltantes antes de construir un modelo de scoring.
- El nivel educativo se codifica como ordinal para que un modelo entienda que "Educación Superior" > "Secundaria".

<a id="retos"></a>
## 11. Retos de Práctica

### 🥉 Reto Básico
Calcula media, mediana, desviación estándar e IQR de `AMT_CREDIT` y `antiguedad_laboral_anios`, e indica si tienen outliers según el criterio IQR.

In [ ]:
# Tu solución al Reto Básico aquí


### 🥈 Reto Medio
Genera la matriz de correlación de Pearson de todas las variables numéricas de `df`, e imputa los valores faltantes de `AMT_INCOME_TOTAL` (numérica) y `OCCUPATION_TYPE` (categórica, si no lo hiciste ya) con las estrategias que consideres más apropiadas.

In [ ]:
# Tu solución al Reto Medio aquí


### 🥇 Reto Avanzado
Construye un flujo completo de preprocesamiento manual sobre `df`: imputa todos los faltantes, codifica `CODE_GENDER` (nominal, One-Hot) y `NAME_EDUCATION_TYPE` (ordinal, ya viste el ejemplo), y escala `AMT_INCOME_TOTAL`, `AMT_CREDIT` y `AMT_ANNUITY` con `StandardScaler`. Deja un DataFrame final `df_listo_para_modelar` sin ningún valor nulo.

In [ ]:
# Tu solución al Reto Avanzado aquí
